# Job Queue with Status Page Example

This notebook demonstrates a job queue system with a comprehensive status page and job management features.

In [1]:
from fasthtml.common import *
import uuid, time, threading
from datetime import datetime
from typing import Dict, Any

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes, btn_behaviors
from cjm_fasthtml_daisyui.components.actions.modal import modal, modal_box, modal_action
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.feedback.alert import alert, alert_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_display.table import table, table_modifiers
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors
from cjm_fasthtml_daisyui.components.data_display.stat import stat, stat_title, stat_value, stats
from cjm_fasthtml_daisyui.components.data_input.text_input import text_input
from cjm_fasthtml_daisyui.components.data_input.select import select
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size, text_color
from cjm_fasthtml_tailwind.utilities.sizing import w, max_w, max_h, container
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, gap
from cjm_fasthtml_tailwind.utilities.effects import shadow
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from cjm_fasthtml_daisyui.core.testing import create_test_app

app, rt = create_test_app(theme=DaisyUITheme.BUSINESS)

# Initialize with history for job tracking
monitor = ProgressMonitor(keep_history=True, history_limit=200)
runner = JobRunner(monitor)

# Job metadata storage
job_metadata: Dict[str, Any] = {}
job_results: Dict[str, Any] = {}

def render_submit_button(disabled=False):
    """Render submit button with appropriate state"""
    btn_classes = combine_classes(
        btn,
        btn_colors.primary,
        btn_behaviors.disabled if disabled else ""
    )
    
    return Button(
        "Submit Job",
        type="submit",
        id="submit-job-btn",
        cls=btn_classes,
        disabled=disabled,
        hx_swap_oob="true"  # Enable out-of-band swap
    )

In [4]:
# Cancellable job runner extension
class CancellableJobRunner(JobRunner):
    def __init__(self, monitor):
        super().__init__(monitor)
        self._stop_events = {}
    
    def start_cancellable(self, job_id, fn, *args, patch_kwargs=None, **kwargs):
        stop_event = threading.Event()
        self._stop_events[job_id] = stop_event
        
        def wrapper():
            try:
                result = fn(stop_event, *args, **kwargs)
                job_results[job_id] = {"status": "success", "data": result}
            except Exception as e:
                job_results[job_id] = {"status": "error", "error": str(e)}
            finally:
                # Clean up stop event when job finishes
                if job_id in self._stop_events:
                    del self._stop_events[job_id]
        
        return self.start(job_id, wrapper, patch_kwargs=patch_kwargs)
    
    def cancel(self, job_id):
        if job_id in self._stop_events:
            # Set the stop event to signal the job to stop
            self._stop_events[job_id].set()
            
            # Mark job as cancelled in metadata
            if job_id in job_metadata:
                job_metadata[job_id]["status"] = "cancelled"
            
            # Mark job as cancelled in results
            job_results[job_id] = {"status": "cancelled"}
            
            # Force mark as completed in monitor so it can be cleared
            # This is a workaround since the monitor doesn't have a native cancel method
            snapshot = monitor.snapshot(job_id)
            if snapshot:
                # Update the monitor's internal state to mark as completed
                # This ensures the job shows as completed and can be cleared
                monitor._jobs[job_id]['completed'] = True
                monitor._jobs[job_id]['overall_progress'] = snapshot['overall_progress']
            
            return True
        return False

# Use cancellable runner
runner = CancellableJobRunner(monitor)

In [5]:
# Various job types
def batch_processing_job(stop_event, batch_size=1000, delay=0.01):
    from tqdm import tqdm
    import time
    
    results = []
    
    # Process in batches
    for batch in range(3):
        desc = f"Batch {batch + 1}/3"
        for i in tqdm(range(batch_size), desc=desc):
            if stop_event.is_set():
                return {"status": "cancelled", "processed": results}
            time.sleep(delay)
            results.append(f"item_{batch}_{i}")
    
    return {"status": "complete", "processed": results}

def data_export_job(stop_event, format_type="csv", records=200):
    from tqdm import tqdm
    import time
    
    # Fetch data
    for _ in tqdm(range(records // 2), desc="Fetching data"):
        if stop_event.is_set():
            return {"status": "cancelled"}
        time.sleep(0.01)
    
    # Format data
    for _ in tqdm(range(records // 4), desc=f"Formatting as {format_type}"):
        if stop_event.is_set():
            return {"status": "cancelled"}
        time.sleep(0.02)
    
    # Write to file
    for _ in tqdm(range(records // 4), desc="Writing to file"):
        if stop_event.is_set():
            return {"status": "cancelled"}
        time.sleep(0.01)
    
    return {"status": "complete", "file": f"export_{format_type}_{records}.{format_type}"}

def model_training_job(stop_event, epochs=10, batch_size=32):
    from tqdm import tqdm
    import time
    import random
    
    metrics = []
    
    for epoch in range(epochs):
        # Training
        for _ in tqdm(range(1000), desc=f"Epoch {epoch+1}/{epochs} - Training"):
            if stop_event.is_set():
                return {"status": "cancelled", "metrics": metrics}
            time.sleep(0.005)
        
        # Validation
        for _ in tqdm(range(200), desc=f"Epoch {epoch+1}/{epochs} - Validation"):
            if stop_event.is_set():
                return {"status": "cancelled", "metrics": metrics}
            time.sleep(0.01)
        
        # Record metrics
        metrics.append({
            "epoch": epoch + 1,
            "loss": random.uniform(0.1, 1.0),
            "accuracy": random.uniform(0.7, 0.99)
        })
    
    return {"status": "complete", "metrics": metrics}

In [6]:
# Job queue management page with HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_page

@rt("/")
def index():
    return create_test_page(
        "Job Queue Manager",
        Div(
            H1("Job Queue Management System", cls=combine_classes(font_size._3xl, font_weight.bold, m.b(6))),
            
            # Job creation panel
            Div(
                H2("Create New Job", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Form(
                    Select(
                        Option("Batch Processing", value="batch"),
                        Option("Data Export", value="export"),
                        Option("Model Training", value="training"),
                        name="type",
                        cls=combine_classes(select, w.full, m.b(3))
                    ),
                    Input(
                        name="name",
                        type="text",
                        placeholder="Job name (optional)",
                        cls=combine_classes(text_input, w.full, m.b(3))
                    ),
                    Div(
                        Button(
                            "Submit Job",
                            type="submit",
                            id="submit-job-btn",
                            cls=combine_classes(btn, btn_colors.primary)
                        ),
                        Button(
                            "Clear Finished",
                            title="Clear completed and cancelled jobs",
                            hx_post="/clear_completed",
                            hx_target="#job-queue",
                            hx_swap="innerHTML",
                            cls=combine_classes(btn, btn_colors.warning, m.l(2))
                        ),
                        cls=combine_classes(flex_display, gap(2))
                    ),
                    hx_post="/create_job",
                    hx_target="#job-queue",
                    hx_swap="innerHTML",
                    # Also trigger stats update after job creation
                    hx_trigger_after_swap="htmx:afterSwap from:body",
                    hx_on_after_request="this.reset()",
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Statistics that auto-refresh only when jobs are running
            Div(
                H2("Queue Statistics", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    hx_get="/stats",
                    hx_trigger="load",  # Only load initially, polling added conditionally
                    hx_swap="outerHTML",  # Use outerHTML to replace the entire element
                    id="stats",
                    cls=combine_classes(stats, shadow(), w.full)
                ),
                cls=str(m.b(6))
            ),
            
            # Job queue table - polling added conditionally
            Div(
                H2("Job Queue", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    hx_get="/queue",
                    hx_trigger="load",  # Only load initially, polling added conditionally
                    hx_swap="innerHTML",
                    id="job-queue",
                    cls=str(overflow.x.auto)
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Job details modal
            Dialog(
                Div(
                    H3("Job Details", cls=combine_classes(font_weight.bold, font_size.lg, m.b(4))),
                    Div(id="job-details-content"),
                    Div(
                        Button(
                            "Close",
                            onclick="this.closest('dialog').close()",
                            cls=combine_classes(btn, btn_sizes.sm)
                        ),
                        cls=str(modal_action)
                    ),
                    cls=combine_classes(modal_box, max_w._4xl)
                ),
                id="job-modal",
                cls=str(modal)
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        )
    )

In [7]:
# API endpoints using FastHTML conventions
@rt("/create_job")
async def create_job(request):
    form = await request.form()
    job_type = form.get('type', 'batch')
    job_name = form.get('name', '')
    
    job_id = str(uuid.uuid4())
    
    # Select job function with appropriate parameters
    job_configs = {
        'batch': (batch_processing_job, {'batch_size': 50, 'delay': 0.005}),
        'export': (data_export_job, {'format_type': 'csv', 'records': 100}),
        'training': (model_training_job, {'epochs': 3, 'batch_size': 32})
    }
    
    job_fn, kwargs = job_configs.get(job_type, (batch_processing_job, {}))
    
    # Store metadata
    job_metadata[job_id] = {
        'id': job_id,
        'name': job_name or f"{job_type.title()} Job",
        'type': job_type,
        'status': 'running',
        'created_at': datetime.now().isoformat(),
        'params': kwargs
    }
    
    # Start job with appropriate throttling
    runner.start_cancellable(
        job_id,
        job_fn,
        **kwargs,
        patch_kwargs={
            "min_delta_pct": 5,
            "min_update_interval": 0.05,
            "emit_initial": True
        }
    )
    
    # Small delay to allow monitor to register the job
    import time
    time.sleep(0.1)  # 100ms delay
    
    # Get queue content with new cell-level polling
    queue_content = queue()
    
    # Always include button update in create_job response (force disabled state)
    button_update = render_submit_button(disabled=True)
    
    # Add stats OOB update
    stats_update = queue_stats()
    stats_update.attrs['id'] = 'stats'
    stats_update.attrs['hx-swap-oob'] = 'true'
    
    # Return combined response
    response = Div(
        queue_content,  # Main content for #job-queue
        button_update,  # Force button to disabled state
        stats_update    # OOB update for stats
    )
    
    return response

In [8]:
@rt("/queue")
def queue():
    jobs = monitor.all()
        
    # Check if any jobs are running
    has_running = False
    running_jobs = []
    
    if not jobs:
        # No jobs - enable the button
        button_update = render_submit_button(disabled=False)
        result = P("No jobs in queue", cls=str(text_color.gray(500)))
        return Div(button_update, result)
    
    rows = []
    for job_id, job in jobs.items():
        meta = job_metadata.get(job_id, {})
        
        # Determine status - check metadata first for cancelled status
        if meta.get('status') == 'cancelled':
            status_text = "Cancelled"
            status_color = badge_colors.error
        elif job['completed']:
            status_text = "Complete"
            status_color = badge_colors.success
        else:
            status_text = "Running"
            status_color = badge_colors.info
            has_running = True  # Found a running job
            running_jobs.append(job_id)  # Track which jobs are running
                
        rows.append(
            Tr(
                Td(job_id[:8] + "...", id=f"job-id-{job_id}"),
                Td(meta.get('name', 'Unknown'), id=f"job-name-{job_id}"),
                Td(meta.get('type', 'unknown').title(), id=f"job-type-{job_id}"),
                Td(
                    Progress(
                        value=str(job['overall_progress']),
                        max="100",
                        cls=combine_classes(progress, progress_colors.primary, w(20))
                    ),
                    id=f"job-progress-{job_id}",
                    # Add polling for this specific cell if job is running
                    hx_get=f"/job_progress?job_id={job_id}" if not job['completed'] and meta.get('status') != 'cancelled' else None,
                    hx_trigger="load delay:500ms" if not job['completed'] and meta.get('status') != 'cancelled' else None,
                    hx_swap="innerHTML"
                ),
                Td(
                    Span(
                        status_text,
                        cls=combine_classes(badge, status_color)
                    ),
                    id=f"job-status-{job_id}",
                    # Add polling for status if job is running
                    hx_get=f"/job_status?job_id={job_id}" if not job['completed'] and meta.get('status') != 'cancelled' else None,
                    hx_trigger="load delay:500ms" if not job['completed'] and meta.get('status') != 'cancelled' else None,
                    hx_swap="innerHTML"
                ),
                Td(
                    Button(
                        "View",
                        hx_get=f"/job_details?job_id={job_id}",
                        hx_target="#job-details-content",
                        hx_swap="innerHTML",
                        onclick="document.getElementById('job-modal').showModal()",
                        cls=combine_classes(btn, btn_sizes.xs, btn_colors.primary)
                    ),
                    Button(
                        "Cancel",
                        hx_post=f"/cancel_job?job_id={job_id}",
                        hx_target="body",  # Target body for OOB swaps
                        hx_swap="none",  # Don't swap anything into body
                        cls=combine_classes(btn, btn_sizes.xs, btn_colors.error, m.l(1))
                    ) if not job['completed'] and meta.get('status') != 'cancelled' else "",
                    id=f"job-actions-{job_id}"
                ),
                id=f"job-row-{job_id}"
            )
        )
    
    # Update button state
    button_update = render_submit_button(disabled=has_running)
    
    # Build the table
    table_content = Table(
        Thead(
            Tr(
                Th("ID"),
                Th("Name"),
                Th("Type"),
                Th("Progress"),
                Th("Status"),
                Th("Actions")
            )
        ),
        Tbody(*rows),
        cls=combine_classes(table, table_modifiers.zebra, w.full)
    )
    
    # Create wrapper div
    wrapper = Div(table_content)
    
    # Add a trigger to refresh stats if jobs are running
    if has_running:
        # Only trigger stats refresh, not table refresh
        wrapper.attrs['hx-trigger'] = "load delay:2s"
        wrapper.attrs['hx-get'] = "/trigger_stats"
        wrapper.attrs['hx-swap'] = "none"
    
    # Return wrapper with button update
    return Div(button_update, wrapper)

In [9]:
@rt("/stats")
def queue_stats():
    
    jobs = monitor.all()
    
    total = len(jobs)
    running = 0
    completed = 0
    cancelled = 0
    
    for job_id, job in jobs.items():
        meta = job_metadata.get(job_id, {})
        if meta.get('status') == 'cancelled':
            cancelled += 1
        elif job['completed']:
            completed += 1
        else:
            running += 1    
    
    stats_content = Div(
        Div(
            Div("Total Jobs", cls=str(stat_title)),
            Div(str(total), cls=str(stat_value)),
            cls=str(stat)
        ),
        Div(
            Div("Running", cls=str(stat_title)),
            Div(str(running), cls=combine_classes(stat_value, text_dui.primary)),
            cls=str(stat)
        ),
        Div(
            Div("Completed", cls=str(stat_title)),
            Div(str(completed), cls=combine_classes(stat_value, text_dui.success)),
            cls=str(stat)
        ),
        Div(
            Div("Cancelled", cls=str(stat_title)),
            Div(str(cancelled), cls=combine_classes(stat_value, text_dui.error)),
            cls=str(stat)
        ),
        id="stats",  # Always include the ID
        cls=combine_classes(stats, shadow(), w.full)
    )
    
    # Add polling trigger only if jobs are running
    if running > 0:
        
        stats_content.attrs['hx-get'] = "/stats"
        stats_content.attrs['hx-trigger'] = "load delay:2s"
        stats_content.attrs['hx-swap'] = "outerHTML"    
    
    return stats_content

In [10]:
@rt("/job_progress")
def job_progress(job_id: str):
    """Update only the progress cell for a specific job"""
    job = monitor.all().get(job_id)
    meta = job_metadata.get(job_id, {})
    
    if not job:
        return ""
    
    # Create the progress element
    progress_elem = Progress(
        value=str(job['overall_progress']),
        max="100",
        cls=combine_classes(progress, progress_colors.primary, w(20))
    )
    
    # If job is still running, continue polling
    if not job['completed'] and meta.get('status') != 'cancelled':
        return Div(
            progress_elem,
            hx_get=f"/job_progress?job_id={job_id}",
            hx_trigger="load delay:500ms",
            hx_swap="outerHTML"
        )
    else:
        # Job completed, stop polling
        return progress_elem

In [11]:
@rt("/job_status")  
def job_status(job_id: str):
    """Update only the status cell for a specific job"""
    job = monitor.all().get(job_id)
    meta = job_metadata.get(job_id, {})
    
    if not job:
        return ""
    
    # Determine status
    if meta.get('status') == 'cancelled':
        status_text = "Cancelled"
        status_color = badge_colors.error
        is_complete = True
    elif job['completed']:
        status_text = "Complete"
        status_color = badge_colors.success
        is_complete = True
    else:
        status_text = "Running"
        status_color = badge_colors.info
        is_complete = False
    
    status_badge = Span(
        status_text,
        cls=combine_classes(badge, status_color)
    )
    
    # If job completed, trigger updates for other cells
    if is_complete:
        # Return the status badge with triggers to update other cells
        return Div(
            status_badge,
            # Hidden div to trigger actions update
            Div(
                hx_get=f"/job_actions_update?job_id={job_id}",
                hx_trigger="load",
                hx_target=f"#job-actions-{job_id}",
                hx_swap="outerHTML",
                style="display:none"
            ),
            # Hidden div to trigger stats update
            Div(
                hx_get="/stats",
                hx_trigger="load",
                hx_target="#stats",
                hx_swap="outerHTML",
                style="display:none"
            ),
            # Always trigger button update check when a job completes
            Div(
                hx_get="/submit_button_update",
                hx_trigger="load",
                hx_target="#submit-job-btn",
                hx_swap="outerHTML",
                style="display:none"
            )
        )
    else:
        # Still running, continue polling
        return Div(
            status_badge,
            hx_get=f"/job_status?job_id={job_id}",
            hx_trigger="load delay:500ms",
            hx_swap="outerHTML"
        )

@rt("/job_actions_update")
def job_actions_update(job_id: str):
    """Update only the actions cell for a completed job"""
    return Td(
        Button(
            "View",
            hx_get=f"/job_details?job_id={job_id}",
            hx_target="#job-details-content",
            hx_swap="innerHTML",
            onclick="document.getElementById('job-modal').showModal()",
            cls=combine_classes(btn, btn_sizes.xs, btn_colors.primary)
        ),
        id=f"job-actions-{job_id}"
    )

@rt("/submit_button_update")
def submit_button_update():
    """Update the submit button based on current job states"""
    all_jobs = monitor.all()
    has_running = any(
        not j['completed'] and job_metadata.get(jid, {}).get('status') != 'cancelled'
        for jid, j in all_jobs.items()
    )
    
    # Return the button with the correct state
    return render_submit_button(disabled=has_running)

In [12]:
@rt("/trigger_stats")
def trigger_stats():
    """Endpoint to trigger stats refresh without returning content"""
    return Div(
        hx_get="/stats",
        hx_trigger="load", 
        hx_target="#stats",
        hx_swap="outerHTML"
    )

In [13]:
@rt("/job_details")
def job_details(job_id: str):
    """Job details with auto-refresh while running"""
    import json
    snapshot = monitor.snapshot(job_id)
    meta = job_metadata.get(job_id, {})
    result = job_results.get(job_id)
    
    if not snapshot:
        return Div("Job not found", cls=combine_classes(alert, alert_colors.error))
    
    # Build progress bars dynamically
    bars = []
    for bar_id, bar in snapshot['bars'].items():
        bars.append(
            Div(
                P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})",
                  cls=str(font_size.sm)),
                Progress(
                    value=str(bar.progress),
                    max="100",
                    cls=combine_classes(progress, progress_colors.accent, w.full)
                ),
                cls=str(m.b(3))
            )
        )
    
    content = Div(
        # Job info
        Div(
            P(f"ID: {job_id}", cls=combine_classes(font_size.xs, text_color.gray(500))),
            P(f"Name: {meta.get('name', 'Unknown')}", cls=str(font_weight.semibold)),
            P(f"Type: {meta.get('type', 'unknown').title()}"),
            P(f"Created: {meta.get('created_at', 'Unknown')}", cls=str(font_size.sm)),
            cls=str(m.b(4))
        ),
        
        # Overall progress
        Div(
            P(f"Overall Progress: {snapshot['overall_progress']:.1f}%",
              cls=combine_classes(font_weight.bold, m.b(2))),
            Progress(
                value=str(snapshot['overall_progress']),
                max="100",
                cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
            ),
        ),
        
        # Individual bars
        Div(
            H4("Task Progress:", cls=combine_classes(font_weight.semibold, m.b(2))),
            *bars
        ) if bars else "",
        
        # Results if available
        Div(
            H4("Results:", cls=combine_classes(font_weight.semibold, m.b(2))),
            Pre(
                json.dumps(result, indent=2),
                cls=combine_classes(
                    bg_dui.base_300, p(3), rounded(),
                    font_size.xs, overflow.auto, max_h(40)
                )
            ),
            cls=str(m.t(4))
        ) if result else "",
        
        # History if available
        Div(
            H4("History:", cls=combine_classes(font_weight.semibold, m.b(2))),
            P(f"{len(snapshot.get('history', []))} updates recorded", cls=str(font_size.sm)),
            cls=str(m.t(4))
        ) if snapshot.get('history') else ""
    )
    
    # Add auto-refresh if job is still running
    if not snapshot['completed']:
        content.attrs['hx-get'] = f"/job_details?job_id={job_id}"
        content.attrs['hx-trigger'] = "load delay:500ms"
        content.attrs['hx-swap'] = "outerHTML"
    
    return content

In [14]:
@rt("/cancel_job")
def cancel_job(job_id: str):
    success = runner.cancel(job_id)
    
    # Instead of refreshing the whole queue, just trigger updates for specific cells
    meta = job_metadata.get(job_id, {})
    job = monitor.all().get(job_id)
    
    if success and job:
        # Return hidden divs that trigger updates for each cell
        # Since the cancel button has hx-swap="none", these triggers will execute
        return Div(
            # Trigger status cell update
            Div(
                hx_get=f"/job_status_cancelled?job_id={job_id}",
                hx_trigger="load",
                hx_target=f"#job-status-{job_id}",
                hx_swap="innerHTML",
                style="display:none"
            ),
            # Trigger actions cell update
            Div(
                hx_get=f"/job_actions_update?job_id={job_id}",
                hx_trigger="load",
                hx_target=f"#job-actions-{job_id}",
                hx_swap="outerHTML",
                style="display:none"
            ),
            # Trigger stats update
            Div(
                hx_get="/stats",
                hx_trigger="load",
                hx_target="#stats",
                hx_swap="outerHTML",
                style="display:none"
            ),
            # Always trigger button update when cancelling
            Div(
                hx_get="/submit_button_update",
                hx_trigger="load",
                hx_target="#submit-job-btn",
                hx_swap="outerHTML",
                style="display:none"
            )
        )
    else:
        # Fallback to full queue refresh if something went wrong
        return queue()

@rt("/job_status_cancelled")
def job_status_cancelled(job_id: str):
    """Update the status cell to show cancelled status"""
    return Span(
        "Cancelled",
        cls=combine_classes(badge, badge_colors.error)
    )

In [15]:
# Optional SSE endpoint for direct streaming if needed
@rt("/stream")
def stream(job_id: str):
    """SSE endpoint - available for direct EventSource usage if needed"""
    return EventStream(
        sse_stream(
            monitor,
            job_id,
            interval=0.05,
            heartbeat=15.0,
            wait_for_start=True,
            start_timeout=10.0
        )
    )

In [16]:
@rt("/clear_completed")
def clear_completed():
    # Get all jobs before clearing
    all_jobs = monitor.all()
    
    # Clear completed jobs from monitor
    monitor.clear_completed(older_than_seconds=0)
    
    # Also manually remove cancelled jobs (in case monitor didn't clear them)
    for job_id in list(all_jobs.keys()):
        meta = job_metadata.get(job_id, {})
        job = all_jobs.get(job_id, {})
        
        # Remove if completed or cancelled
        if job.get('completed') or meta.get('status') == 'cancelled':
            # Remove from monitor if still there
            if job_id in monitor._jobs:
                del monitor._jobs[job_id]
            
            # Clean up metadata
            if job_id in job_metadata:
                del job_metadata[job_id]
            
            # Clean up results
            if job_id in job_results:
                del job_results[job_id]
    
    # Return updated queue
    return queue()

In [17]:
# Start server for Jupyter
from cjm_fasthtml_daisyui.core.testing import start_test_server
from fasthtml.jupyter import HTMX
from IPython.display import display

server = start_test_server(app)
display(HTMX())

In [18]:
# Stop server when done
server.stop()